In [1]:
import torch
from typing import List, Tuple
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, LlamaForCausalLM
class LLM(torch.nn.Module):
    def __init__(self, model_name='HuggingFaceTB/SmolLM-1.7B-Instruct', device: str = 'cuda:0'):
        super().__init__()
        self.device = device
        self.bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        
        # Load model and tokenizer using bitsandbytes nf4 bit quantization and use accelerate device mapping to distribute the model across all available devices.
        self.model: LlamaForCausalLM = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map='auto', quantization_config=self.bnb_config)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, torch_dtype=torch.float16, padding=True, device_map='auto')
        
    def get_embeddings(self, system: str = None, prompt: str = None, token_ids: List[int] = None) -> torch.Tensor:
        """Your non-standard .generate"""
        assert prompt or token_ids, "A text prompt or token_ids must be passed to get_input_embeddings"
        
        if prompt:
            messages = [{"role": "system", "content": system} if system else None, {"role": "user", "content": prompt}]
            print(messages)
            inputs = self.tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt")
        else:
            inputs = token_ids
        with torch.no_grad():
            embs = self.model.get_input_embeddings()(inputs)
            
        return embs
        
    def generate_from_embeddings(self, text_embeddings, grad=True, max_new_tokens=10) -> Tuple[str, float]:
        next_token_ids = torch.tensor(())
        next_token_probs = torch.tensor(())
        n_tokens = 0
        stop_token = False
        while n_tokens < max_new_tokens and not stop_token: # n_tokens
            if not grad:
                with torch.no_grad():
                    next_token_embs = self.model.forward(inputs_embeds=text_embeddings)
            else:
                next_token_embs = self.model.forward(inputs_embeds=text_embeddings)
            logit =  torch.max(next_token_embs.logits[:, -1, :], dim=-1)
            
            # Update the logits
            device = logit.indices.device
            next_token_ids = torch.cat([next_token_ids.to(device), logit.indices], dim=0)
            next_token_probs = torch.cat([next_token_probs.to(device), logit.values], dim=0)
            # Get embeddings of the new token and add a new dimension (to 3d like text_embeddings is)
            new_embeddings = self.get_embeddings(token_ids=logit.indices).unsqueeze(0)
            
            # Add the new token embeddings to the end of the previous tokens embeddings
            text_embeddings = torch.cat([text_embeddings, new_embeddings], dim=1)
            
            # Check for eos token
            if logit.indices == self.tokenizer.eos_token_id:
                stop_token = True
                
        return next_token_ids.to(int), next_token_probs

/home/jonathan/projects/primaite/PrimAITE/src/primaite/agents/git/aegis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = LLM()

In [3]:
prompt = "I am trying to find the cause of my cough."

In [4]:
embs = llm.get_embeddings(prompt)

In [5]:
token_ids, probs = llm.generate_from_embeddings(embs, max_new_tokens=100)

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)
2024-08-07 09:22:13.694336: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-07 09:22:14.361795: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [6]:
print(llm.tokenizer.decode(token_ids), end='')

<|im_start|>assistant
The cause of your cough is likely due to a viral infection, such as a cold or flu. Viral infections are common and can cause a variety of symptoms, including a cough. It is important to rest and drink plenty of fluids to help your body fight off the infection. If your cough persists for more than a week or is accompanied by other symptoms such as fever, shortness of breath, or chest pain, it is important to seek medical attention.<|im_end|>